In [1]:
import time
import heapq
import random
import sys
from typing import List, Set, Tuple, Dict, Optional
from dataclasses import dataclass
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
from IPython.display import HTML, display # Colabでの動画


# =============================================================================
# 1. データ構造
# =============================================================================

@dataclass
class Operation:
    job: int
    machine: int
    duration: int
    op_index: int

    def __hash__(self): return hash((self.job, self.op_index))
    def __eq__(self, other):
        if not isinstance(other, Operation): return NotImplemented
        return self.job == other.job and self.op_index == other.op_index
    def __lt__(self, other):
        if not isinstance(other, Operation): return NotImplemented
        return (self.job, self.op_index) < (other.job, other.op_index)
    def __repr__(self): return f"Op({self.job},{self.op_index})"

@dataclass
class Job:
    operations: List[Operation]

class DisjunctiveGraph:
    def __init__(self):
        self.selection = {}

    def select_arc(self, op_i: Operation, op_j: Operation, direction: bool):
        key = (op_i, op_j) if op_i < op_j else (op_j, op_i)
        self.selection[key] = direction if op_i < op_j else not direction

    def is_selected(self, op_i: Operation, op_j: Operation) -> Optional[bool]:
        key = (op_i, op_j) if op_i < op_j else (op_j, op_i)
        if key not in self.selection: return None
        result = self.selection[key]
        return result if op_i < op_j else not result

    def copy(self):
        new_graph = DisjunctiveGraph()
        new_graph.selection = self.selection.copy()
        return new_graph

# =============================================================================
# 2. 問題データ (FT10)
# =============================================================================
class ProblemLoader:
    """様々な形式の問題データを生成・読み込みするクラス"""

    @staticmethod
    def parse_text_format(text_data: str) -> List[Job]:
        """
        OR-Library形式 (Lawrenceなど) のテキストデータをパースする
        形式:
        #jobs #machines
        machine_0 duration_0 machine_1 duration_1 ...
        """
        lines = text_data.strip().split('\n')
        # コメント行などをスキップして最初の有効な行を探す
        header = None
        start_idx = 0
        for i, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) >= 2 and parts[0].isdigit():
                header = parts
                start_idx = i + 1
                break

        if not header:
            raise ValueError("Invalid data format")

        # n_jobs = int(header[0]) # 未使用だが形式確認用
        # n_machines = int(header[1])

        jobs = []
        job_id = 0
        for i in range(start_idx, len(lines)):
            line = lines[i].strip()
            if not line: continue
            parts = list(map(int, line.split()))

            ops = []
            # (machine, duration) のペアになっている
            for op_idx in range(0, len(parts), 2):
                m = parts[op_idx]
                d = parts[op_idx+1]
                ops.append(Operation(job=job_id, machine=m, duration=d, op_index=op_idx//2))

            jobs.append(Job(operations=ops))
            job_id += 1

        return jobs

    @staticmethod
    def create_ft06() -> List[Job]:
        """FT06 (6 jobs, 6 machines) - Optimal: 55"""
        # (machine, duration)
        data = [
            [(2, 1), (0, 3), (1, 6), (3, 7), (5, 3), (4, 6)],
            [(1, 8), (2, 5), (4, 10), (5, 10), (0, 10), (3, 4)],
            [(2, 5), (3, 4), (5, 8), (0, 9), (1, 1), (4, 7)],
            [(1, 5), (0, 5), (2, 5), (3, 3), (4, 8), (5, 9)],
            [(2, 9), (1, 3), (4, 5), (5, 4), (0, 3), (3, 1)],
            [(1, 3), (3, 3), (5, 9), (0, 10), (4, 4), (2, 1)]
        ]
        return ProblemLoader._convert_matrix_to_jobs(data)

    @staticmethod
    def create_ft10() -> List[Job]:
        """FT10 (10 jobs, 10 machines) - Optimal: 930"""
        data = [
            [(0, 29), (1, 78), (2, 9), (3, 36), (4, 49), (5, 11), (6, 62), (7, 56), (8, 44), (9, 21)],
            [(0, 43), (2, 90), (4, 75), (9, 11), (3, 69), (1, 28), (6, 46), (5, 46), (7, 72), (8, 30)],
            [(1, 91), (0, 85), (3, 39), (2, 74), (8, 90), (5, 10), (7, 12), (6, 89), (9, 45), (4, 33)],
            [(1, 81), (2, 95), (0, 71), (4, 99), (6, 9), (8, 52), (7, 85), (3, 98), (9, 22), (5, 43)],
            [(2, 14), (0, 6), (1, 22), (5, 61), (3, 26), (4, 69), (8, 21), (7, 49), (9, 72), (6, 53)],
            [(2, 84), (1, 2), (5, 52), (3, 95), (8, 48), (9, 72), (0, 47), (6, 65), (4, 6), (7, 25)],
            [(1, 46), (0, 37), (3, 61), (2, 13), (6, 32), (5, 21), (9, 32), (8, 89), (7, 30), (4, 55)],
            [(2, 31), (0, 86), (1, 46), (5, 74), (4, 32), (6, 88), (8, 19), (9, 48), (7, 36), (3, 79)],
            [(0, 76), (1, 69), (3, 76), (5, 51), (2, 85), (9, 11), (6, 40), (7, 89), (4, 26), (8, 74)],
            [(1, 85), (0, 13), (2, 61), (6, 7), (8, 64), (9, 76), (5, 47), (3, 52), (4, 90), (7, 45)]
        ]
        return ProblemLoader._convert_matrix_to_jobs(data)


    @staticmethod
    def create_ft08() -> List[Job]:
        """FT10 (8 jobs, 8 machines) - Optimal: """

        data = [
          [(0, 5), (1, 12), (2, 8), (3, 20), (4, 15), (5, 6), (6, 9), (7, 10)],
          [(3, 10), (2, 15), (0, 8), (1, 22), (5, 18), (4, 12), (7, 7), (6, 14)],
          [(2, 20), (5, 8), (1, 10), (0, 15), (4, 7), (3, 9), (6, 11), (7, 13)],
          [(1, 14), (0, 6), (3, 12), (2, 18), (6, 10), (5, 5), (7, 16), (4, 9)],
          [(4, 10), (6, 12), (7, 8), (1, 15), (0, 20), (2, 7), (3, 11), (5, 14)],
          [(5, 8), (4, 18), (6, 5), (7, 12), (1, 9), (0, 14), (2, 10), (3, 15)],
          [(6, 15), (7, 10), (5, 12), (4, 8), (2, 20), (3, 6), (1, 18), (0, 7)],
          [(7, 9), (3, 14), (4, 16), (6, 11), (5, 8), (2, 13), (0, 10), (1, 12)]
        ]
        return ProblemLoader._convert_matrix_to_jobs(data)


    @staticmethod
    def create_ft20() -> List[Job]:
        """FT10 (20 jobs, 5 machines) - Optimal: """
        data = [
          [(1, 12), (0, 34), (2, 21), (4, 11), (3, 56)], # Job 0
          [(4, 87), (2, 22), (0, 43), (1, 54), (3, 21)],
          [(0, 23), (3, 12), (1, 65), (2, 43), (4, 32)],
          [(2, 12), (0, 88), (4, 32), (1, 66), (3, 10)],
          [(1, 45), (4, 32), (0, 98), (3, 43), (2, 12)],
          [(3, 12), (2, 43), (4, 11), (0, 32), (1, 54)],
          [(0, 21), (1, 76), (2, 32), (3, 11), (4, 43)],
          [(2, 34), (0, 21), (3, 54), (1, 23), (4, 12)],
          [(4, 65), (1, 43), (0, 21), (3, 32), (2, 11)],
          [(3, 21), (4, 54), (2, 32), (0, 12), (1, 43)],
          [(1, 12), (0, 34), (2, 21), (4, 11), (3, 56)], # Job 10 (Job0と同じパターン)
          [(4, 87), (2, 22), (0, 43), (1, 54), (3, 21)],
          [(0, 23), (3, 12), (1, 65), (2, 43), (4, 32)],
          [(2, 12), (0, 88), (4, 32), (1, 66), (3, 10)],
          [(1, 45), (4, 32), (0, 98), (3, 43), (2, 12)],
          [(3, 12), (2, 43), (4, 11), (0, 32), (1, 54)],
          [(0, 21), (1, 76), (2, 32), (3, 11), (4, 43)],
          [(2, 34), (0, 21), (3, 54), (1, 23), (4, 12)],
          [(4, 65), (1, 43), (0, 21), (3, 32), (2, 11)],
          [(3, 21), (4, 54), (2, 32), (0, 12), (1, 43)]
        ]
        return ProblemLoader._convert_matrix_to_jobs(data)


    @staticmethod
    def create_la03() -> List[Job]:
        """LA01 (10 jobs, 5 machines) - Optimal: 597"""
        # LA01〜LA40はデータ量が多いので、この文字列形式でパースするのが一番効率的です
        la03_text = """
        10 5
        1 23 2 45 0 82 4 84 3 38
        2 21 1 29 0 18 4 41 3 50
        2 38 3 54 4 16 0 52 1 52
        4 37 0 54 2 74 1 62 3 57
        4 57 0 81 1 61 3 68 2 30
        4 81 0 79 1 89 2 89 3 11
        3 33 2 20 0 91 4 20 1 66
        4 24 1 84 0 32 2 55 3  8
        4 56 0  7 3 54 2 64 1 39
        4 40 1 83 0 19 2  8 3  7

        """
        return ProblemLoader.parse_text_format(la03_text)

    @staticmethod
    def _convert_matrix_to_jobs(data) -> List[Job]:
        jobs = []
        for j_idx, job_data in enumerate(data):
            ops = []
            for op_idx, (m, d) in enumerate(job_data):
                ops.append(Operation(job=j_idx, machine=m, duration=d, op_index=op_idx))
            jobs.append(Job(operations=ops))
        return jobs


# =============================================================================
# 3. Carlier-Pinson Solver (Performance Tuned)
# =============================================================================

class CarlierPinsonSolver:
    def __init__(self, jobs: List[Job]):
        self.jobs = jobs
        self.n_jobs = len(jobs)
        self.n_machines = max(op.machine for job in jobs for op in job.operations) + 1
        self.all_operations = [op for job in jobs for op in job.operations]

        self.machine_ops = [[] for _ in range(self.n_machines)]
        for op in self.all_operations:
            self.machine_ops[op.machine].append(op)

        self.best_makespan = float('inf')
        self.best_solution = None
        self.nodes_explored = 0
        self.solution_history = [] # アニメーション用に履歴を保存するリスト

    # --- Heads/Tails Calculation ---
    def compute_heads_tails(self, graph: DisjunctiveGraph) -> Optional[Tuple[Dict, Dict]]:
        heads = {op: 0 for op in self.all_operations}
        tails = {op: 0 for op in self.all_operations}

        successors = {op: [] for op in self.all_operations}
        predecessors = {op: [] for op in self.all_operations}

        # Job sequence
        for job in self.jobs:
            for i in range(len(job.operations) - 1):
                curr, next_op = job.operations[i], job.operations[i + 1]
                successors[curr].append(next_op)
                predecessors[next_op].append(curr)

        # Machine selections
        for machine_id in range(self.n_machines):
            ops = self.machine_ops[machine_id]
            for i in range(len(ops)):
                for j in range(i + 1, len(ops)):
                    sel = graph.is_selected(ops[i], ops[j])
                    if sel is True:
                        successors[ops[i]].append(ops[j])
                        predecessors[ops[j]].append(ops[i])
                    elif sel is False:
                        successors[ops[j]].append(ops[i])
                        predecessors[ops[i]].append(ops[j])

        # Heads (Topological Sort)
        in_degree = {op: len(predecessors[op]) for op in self.all_operations}
        queue = [op for op in self.all_operations if in_degree[op] == 0]
        processed_count = 0
        while queue:
            current = queue.pop(0)
            processed_count += 1
            for next_op in successors[current]:
                heads[next_op] = max(heads[next_op], heads[current] + current.duration)
                in_degree[next_op] -= 1
                if in_degree[next_op] == 0:
                    queue.append(next_op)

        if processed_count < len(self.all_operations): return None # Cycle

        # Tails
        out_degree = {op: len(successors[op]) for op in self.all_operations}
        queue = [op for op in self.all_operations if out_degree[op] == 0]
        while queue:
            current = queue.pop(0)
            for prev_op in predecessors[current]:
                tails[prev_op] = max(tails[prev_op], tails[current] + current.duration)
                out_degree[prev_op] -= 1
                if out_degree[prev_op] == 0:
                    queue.append(prev_op)

        return heads, tails

    # --- Lower Bound (Preemptive Schrage) ---
    def compute_lower_bound_machine(self, machine_id: int, heads: Dict, tails: Dict) -> int:
        ops = self.machine_ops[machine_id]
        if not ops: return 0

        unscheduled = sorted(ops, key=lambda x: heads[x])
        ready = []

        current_time = heads[unscheduled[0]] if unscheduled else 0
        max_lateness = 0
        rem_p = {op: op.duration for op in ops}

        active_op = None
        next_idx = 0

        while next_idx < len(unscheduled) or ready or active_op:
            while next_idx < len(unscheduled) and heads[unscheduled[next_idx]] <= current_time:
                op = unscheduled[next_idx]
                heapq.heappush(ready, (-tails[op], op))
                next_idx += 1

            if active_op and ready:
                best_ready_tail = -ready[0][0]
                if best_ready_tail > tails[active_op]:
                    heapq.heappush(ready, (-tails[active_op], active_op))
                    active_op = None

            if not active_op and ready:
                _, active_op = heapq.heappop(ready)

            if active_op:
                dt_finish = rem_p[active_op]
                dt_release = heads[unscheduled[next_idx]] - current_time if next_idx < len(unscheduled) else float('inf')
                dt = min(dt_finish, dt_release)

                current_time += dt
                rem_p[active_op] -= dt

                if rem_p[active_op] <= 1e-9:
                    max_lateness = max(max_lateness, current_time + tails[active_op])
                    active_op = None
            else:
                if next_idx < len(unscheduled):
                    current_time = heads[unscheduled[next_idx]]
                else:
                    break
        return max_lateness

    # --- Jackson Schedule (For Critical Set) ---
    def jackson_schedule(self, ops: List[Operation], heads: Dict, tails: Dict):
        if not ops: return 0, [], None

        unscheduled = set(ops)
        available = []
        current_time = min(heads[op] for op in ops)
        schedule = []
        makespan = 0

        while unscheduled or available:
            if not available and unscheduled:
                min_r = min(heads[op] for op in unscheduled)
                current_time = max(current_time, min_r)

            to_remove = []
            for op in unscheduled:
                if heads[op] <= current_time:
                    heapq.heappush(available, (-tails[op], op))
                    to_remove.append(op)
            for op in to_remove: unscheduled.remove(op)

            if available:
                neg_q, op = heapq.heappop(available)
                q = -neg_q

                start = current_time
                end = start + op.duration
                cmp_val = end + q
                schedule.append((op, start, end, cmp_val))
                makespan = max(makespan, cmp_val)
                current_time = end

        critical_candidates = [item for item in schedule if item[3] == makespan]
        if not critical_candidates: return makespan, [], None

        last_item = critical_candidates[0]
        schedule.sort(key=lambda x: x[1])

        idx = schedule.index(last_item)
        start_idx = idx
        current_start = last_item[1]

        for i in range(idx - 1, -1, -1):
            prev_item = schedule[i]
            if prev_item[2] == current_start:
                start_idx = i
                current_start = prev_item[1]
            else:
                if prev_item[2] < current_start: break

        critical_set = [schedule[i][0] for i in range(start_idx, idx + 1)]
        critical_op = max(critical_set, key=lambda x: tails[x])
        return makespan, critical_set, critical_op

    # --- Propositions 6 & 7 Logic ---
    def evaluate_input_output(self, clique, heads, tails, L):
        # min_q calculation: safe approximation
        # The logic: if i is first, completion >= r_i + sum_p + min_{k \in K \setminus i} q_k
        # We use min_{k \in K} q_k which is <= true value, so it is a Safe Lower Bound.

        min_r = min(heads[op] for op in clique)
        min_q = min(tails[op] for op in clique)
        sum_p = sum(op.duration for op in clique)

        I_set = set(clique) # Candidates for Input (First)
        O_set = set(clique) # Candidates for Output (Last)

        for op in clique:
            # Check First: r_i + P_K + min_q > L => i cannot be first
            if heads[op] + sum_p + min_q > L:
                I_set.discard(op)

            # Check Last: min_r + P_K + q_i > L => i cannot be last
            if min_r + sum_p + tails[op] > L:
                O_set.discard(op)

        return I_set, O_set

    def apply_propositions(self, graph, L):
        active = True
        loop_cnt = 0
        while active and loop_cnt < 10: # Limit iterations
            active = False
            loop_cnt += 1

            res = self.compute_heads_tails(graph)
            if res is None: return False, float('inf')
            heads, tails = res

            machine_lbs = []
            for m in range(self.n_machines):
                lb = self.compute_lower_bound_machine(m, heads, tails)
                machine_lbs.append(lb)
                if lb > L: return False, lb

                ops = self.machine_ops[m]
                if len(ops) < 2: continue

                # Prop 5: Immediate Selection
                for i in range(len(ops)):
                    for j in range(i+1, len(ops)):
                        if graph.is_selected(ops[i], ops[j]) is not None: continue

                        if heads[ops[i]] + ops[i].duration + ops[j].duration + tails[ops[j]] > L:
                            graph.select_arc(ops[j], ops[i], True)
                            active = True
                        elif heads[ops[j]] + ops[j].duration + ops[i].duration + tails[ops[i]] > L:
                            graph.select_arc(ops[i], ops[j], True)
                            active = True

                # Prop 6 & 7: Input/Output Checks (Re-enabled for performance)
                _, K, _ = self.jackson_schedule(ops, heads, tails)
                if K and len(K) > 1:
                    I_K, O_K = self.evaluate_input_output(K, heads, tails, L)

                    # Fix Not First
                    not_first = [op for op in K if op not in I_K]
                    if len(not_first) == len(K) - 1:
                        first_op = list(I_K)[0]
                        for other in K:
                            if other != first_op and graph.is_selected(first_op, other) is None:
                                graph.select_arc(first_op, other, True)
                                active = True

                    # Fix Not Last
                    not_last = [op for op in K if op not in O_K]
                    if len(not_last) == len(K) - 1:
                        last_op = list(O_K)[0]
                        for other in K:
                            if other != last_op and graph.is_selected(other, last_op) is None:
                                graph.select_arc(other, last_op, True)
                                active = True

            if machine_lbs and max(machine_lbs) > L:
                return False, max(machine_lbs)

        res = self.compute_heads_tails(graph)
        if res is None: return False, float('inf')
        heads, tails = res
        final_lbs = [self.compute_lower_bound_machine(m, heads, tails) for m in range(self.n_machines)]
        return True, max(final_lbs) if final_lbs else 0

    # --- Heuristics ---
    def compute_greedy_ub(self) -> int:
        machine_free = [0] * self.n_machines
        job_free = [0] * self.n_jobs
        ops_by_job = {i: 0 for i in range(self.n_jobs)}
        count = 0
        while count < len(self.all_operations):
            candidates = []
            for j in range(self.n_jobs):
                if ops_by_job[j] < len(self.jobs[j].operations):
                    op = self.jobs[j].operations[ops_by_job[j]]
                    start = max(job_free[j], machine_free[op.machine])
                    candidates.append((start, op))
            if not candidates: break
            candidates.sort(key=lambda x: x[0])
            start, op = candidates[0]
            machine_free[op.machine] = start + op.duration
            job_free[op.job] = start + op.duration
            ops_by_job[op.job] += 1
            count += 1
        return max(job_free)

    def is_complete_selection(self, graph):
        for m in range(self.n_machines):
            ops = self.machine_ops[m]
            for i in range(len(ops)):
                for j in range(i + 1, len(ops)):
                    if graph.is_selected(ops[i], ops[j]) is None: return False
        return True

    def compute_makespan(self, graph):
        res = self.compute_heads_tails(graph)
        if res is None: return float('inf')
        heads, _ = res
        return max(heads[op] + op.duration for op in self.all_operations)

    def choose_disjunction(self, graph, heads, tails) -> Optional[Tuple[Operation, Operation]]:
        candidates = []
        for m in range(self.n_machines):
            ops = self.machine_ops[m]
            if len(ops) < 2: continue

            pairs = []
            for i in range(len(ops)):
                for j in range(i + 1, len(ops)):
                    if graph.is_selected(ops[i], ops[j]) is None:
                        # Heuristic: Sum of durations (impact)
                        score = ops[i].duration + ops[j].duration
                        pairs.append((score, ops[i], ops[j]))

            if pairs:
                # Add randomness to break ties and local optima
                # Primary sort: Score (Desc), Secondary: Random
                pairs.sort(key=lambda x: (x[0], random.random()), reverse=True)
                best_pair = (pairs[0][1], pairs[0][2])

                lb = self.compute_lower_bound_machine(m, heads, tails)
                candidates.append((lb, best_pair))

        # Branch on the machine with highest LB (Bottleneck)
        candidates.sort(key=lambda x: x[0], reverse=True)
        if candidates:
            return candidates[0][1]
        return None

    def create_animation(self):
        """
        保存された solution_history を使ってアニメーションを作成し、HTMLオブジェクトを返す
        """
        if not self.solution_history:
            print("表示する履歴がありません。")
            return None

        print(f"アニメーション生成中... (フレーム数: {len(self.solution_history)})")

        # 描画の基本設定
        fig, ax = plt.subplots(figsize=(12, 6))
        colors = list(mcolors.TABLEAU_COLORS.values())

        # グラフの最大幅（時間軸）を固定するために、最も悪い解のMakespanを取得
        max_time_limit = self.solution_history[0][0] * 1.1

        def update(frame):
            ax.clear()
            makespan, graph = self.solution_history[frame]

            # スケジュール計算
            res = self.compute_heads_tails(graph)
            if res is None: return
            heads, _ = res

            # 描画
            for m in range(self.n_machines):
                for op in self.machine_ops[m]:
                    start = heads[op]
                    color = colors[op.job % len(colors)]

                    ax.barh(m, op.duration, left=start, height=0.6,
                            color=color, edgecolor='black', alpha=0.8)

                    # 文字サイズ調整
                    ax.text(start + op.duration/2, m, f"J{op.job}",
                            ha='center', va='center', color='white',
                            fontsize=8, fontweight='bold')

            ax.set_yticks(range(self.n_machines))
            ax.set_yticklabels([f"M{m}" for m in range(self.n_machines)])
            ax.set_xlabel("Time")
            ax.set_title(f"Optimization Process: Step {frame+1}/{len(self.solution_history)} (Makespan: {makespan})")
            ax.invert_yaxis()
            ax.grid(True, axis='x', linestyle='--', alpha=0.5)
            ax.set_xlim(0, max_time_limit) # 軸のスケールを固定

        # アニメーション作成
        anim = animation.FuncAnimation(fig, update, frames=len(self.solution_history), interval=1000)
        plt.close(fig) # 余分な静止画プロットを防ぐ

        return HTML(anim.to_jshtml())



    # --- Main Solve ---
    def solve(self, time_limit: int = 600):
        start_time = time.time()
        print(f"Solving ft10 with Time Limit: {time_limit}s")

        self.best_makespan = float('inf')
        self.best_solution = None
        self.solution_history = [] # 履歴リセット

        # 1. Initialize with Greedy UB
        heuristic_ub = self.compute_greedy_ub()
        print(f"Heuristic UB (Reference): {heuristic_ub}")
        self.best_makespan = heuristic_ub # Set heuristic as initial best

        root_graph = DisjunctiveGraph()

        # 2. Preprocessing at Root (Enable strong inference by passing finite UB)
        # Using heuristic_ub helps prune impossible regions early.
        print("Preprocessing root node...")
        self.apply_propositions(root_graph, heuristic_ub)

        res = self.compute_heads_tails(root_graph)
        if res is None: return float('inf'), None
        heads, tails = res
        root_lb = max(self.compute_lower_bound_machine(m, heads, tails) for m in range(self.n_machines))
        print(f"Initial Lower Bound: {root_lb}") # Should be 894

        stack = [(root_graph, root_lb)]
        self.nodes_explored = 0

        while stack:
            if time.time() - start_time > time_limit:
                print("\n" + "!"*40)
                print(" Time Limit Exceeded!")
                break

            graph, node_lb = stack.pop()

            self.nodes_explored += 1
            if self.nodes_explored % 1000 == 0:
                print(f"\rNodes: {self.nodes_explored}, Current Best: {self.best_makespan}, Stack: {len(stack)}", end="")

            # Pruning: STRICTLY Greater than best makespan means no improvement
            # Allowing == to continue for finding optimal
            if node_lb >= self.best_makespan:
                continue

            # Inference
            feasible, refined_lb = self.apply_propositions(graph, self.best_makespan - 1) # Look for strictly better
            if not feasible: continue
            if refined_lb >= self.best_makespan: continue

            # Check Solution
            if self.is_complete_selection(graph):
                mk = self.compute_makespan(graph)
                if mk < self.best_makespan:
                    print(f"\n★ New Best Found: {mk} (Time: {time.time()-start_time:.1f}s)")
                    self.best_makespan = mk
                    self.best_solution = graph

                    # 【重要】履歴に追加 (グラフはコピーして保存)
                    self.solution_history.append((mk, graph.copy()))

                    # If we hit the known optimal, we can stop or keep verifying
                    #if mk <= 930:
                    #    print("Optimal solution 930 reached!")
                    #    break
                continue

            # Branching
            res = self.compute_heads_tails(graph)
            if res is None: continue
            heads, tails = res

            disj = self.choose_disjunction(graph, heads, tails)
            if disj is None: continue
            op_i, op_j = disj

            children = []
            for direction in [True, False]:
                g_child = graph.copy()
                g_child.select_arc(op_i, op_j, direction)

                res_c = self.compute_heads_tails(g_child)
                if res_c is None: continue
                h_c, t_c = res_c

                lb_c = max(self.compute_lower_bound_machine(m, h_c, t_c) for m in range(self.n_machines))

                if lb_c < self.best_makespan:
                    children.append((g_child, lb_c))

            # DFS Best-First
            children.sort(key=lambda x: x[1], reverse=True)
            for child in children:
                stack.append(child)

        print("\n" + "="*50)
        if self.best_solution is None:
            print("Result: No solution found (or Heuristic was best).")
        else:
            print(f"Result: Best Makespan = {self.best_makespan}")
            if self.best_makespan == 930:
                print("Congratulations! Optimal solution found.")
            return self.best_makespan, self.best_solution
        return float('inf'), None

    def draw_gantt_chart(self, graph):
        if graph is None: return
        res = self.compute_heads_tails(graph)
        if res is None: return
        heads, _ = res
        makespan = self.compute_makespan(graph)

        fig, ax = plt.subplots(figsize=(12, 6))
        colors = list(mcolors.TABLEAU_COLORS.values())

        for m in range(self.n_machines):
            for op in self.machine_ops[m]:
                start = heads[op]
                color = colors[op.job % len(colors)]
                ax.barh(m, op.duration, left=start, height=0.6,
                        color=color, edgecolor='black', alpha=0.8)
                ax.text(start + op.duration/2, m, f"J{op.job}",
                        ha='center', va='center', color='white', fontsize=8, fontweight='bold')

        ax.set_yticks(range(self.n_machines))
        ax.set_yticklabels([f"M{m}" for m in range(self.n_machines)])
        ax.set_xlabel("Time")
        ax.set_title(f"FT10 Gantt Chart (Makespan: {makespan})")
        ax.invert_yaxis()
        ax.grid(True, axis='x', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

# =============================================================================
# 実行
# =============================================================================
if __name__ == "__main__":
    try:
        %matplotlib inline
    except:
        pass
    
    print("1:ベンチマーク(ft10)")
    print("2:ベンチマーク(ft06)")
    print("3:8x8 Custom)")
    print("4:20x5 Custom)")
    print("5:ベンチマーク(la03)")
    
    try:
        switch = int(input("選択: "))
    except ValueError:
        switch = 1 # default to manual

    if switch == 1:
        print('ft10が選択されました')
        jobs = ProblemLoader.create_ft10()
    elif switch == 2:
        print('06問題が選択されました')
        jobs = ProblemLoader.create_ft06()
    elif switch == 3:
        print('08問題が選択されました')
        jobs = ProblemLoader.create_ft08()
    elif switch == 4:
        print('ft20が選択されました')
        jobs = ProblemLoader.create_ft20()
    elif switch == 5:
        print('la01が選択されました')
        jobs = ProblemLoader.create_la03()
    
    #jobs = create_ft10()
    solver = CarlierPinsonSolver(jobs)

    # 計算実行 (例えば10分間)
    mk, sol = solver.solve(time_limit=3600)

    # 計算終了後、アニメーションを表示
    if solver.solution_history:
        print("\n最適化プロセスの再生を開始します...")
        display(solver.create_animation())
    else:
        print("表示する履歴がありません。")


1:ベンチマーク(ft10)
2:ベンチマーク(ft06)
3:8x8 Custom)
4:20x5 Custom)
5:ベンチマーク(la03)
ft10が選択されました
Solving ft10 with Time Limit: 3600s
Heuristic UB (Reference): 1262
Preprocessing root node...
Initial Lower Bound: 808

★ New Best Found: 1052 (Time: 1.2s)

★ New Best Found: 1026 (Time: 1.4s)

★ New Best Found: 1020 (Time: 1.4s)

★ New Best Found: 1019 (Time: 1.4s)

★ New Best Found: 1014 (Time: 1.9s)

★ New Best Found: 1010 (Time: 2.0s)
Nodes: 1000, Current Best: 1010, Stack: 111
★ New Best Found: 1008 (Time: 5.7s)

★ New Best Found: 1007 (Time: 7.4s)
Nodes: 2000, Current Best: 1007, Stack: 106
★ New Best Found: 1004 (Time: 10.3s)

★ New Best Found: 1001 (Time: 13.0s)

★ New Best Found: 999 (Time: 13.1s)
Nodes: 3000, Current Best: 999, Stack: 68
★ New Best Found: 995 (Time: 16.1s)

★ New Best Found: 994 (Time: 16.3s)
Nodes: 4000, Current Best: 994, Stack: 60
★ New Best Found: 989 (Time: 22.6s)

★ New Best Found: 988 (Time: 22.8s)
Nodes: 5000, Current Best: 988, Stack: 61
★ New Best Found: 986 (Time